(course-antiviral-conformational-engineering)=
# Module 27: Conformational Engineering

Study the N3 inhibitor in the original
[SARS-CoV-2 main-protease complex 6LU7](https://www.rcsb.org/structure/6LU7).
N3 spans six deposited groups in **chain C**; selecting generic small molecules
would miss this heterogeneous peptide-like inhibitor. First measure what the
input actually supports before attempting conformational editing.

> **🎯 Learning Outcomes**
> - Extract N3 by explicit source labels.
> - Find four-distinct-atom paths in declared connectivity.
> - Separate measured torsions from chemically justified rotatable bonds.

In [1]:
import numpy as np
import molsysmt as msm
from molsysmt import pyunitwizard as puw

complex_molsys = msm.convert(
    msm.systems['SARS-CoV-2 main protease']['6lu7.pdb'], get_missing_bonds=False)
molsys = msm.extract(complex_molsys, selection='chain_id=="C"')
original_coordinates = puw.get_value(msm.get(molsys, coordinates=True), to_unit='nm').copy()

(course-antiviral-conformational-engineering-measurement)=
## 1. Measuring Declared Covalent Paths

Keep deposited CONECT connectivity without automatically inventing complete
ligand chemistry. The public path tool returns walks, including reversals and
revisited atoms: filter to four distinct atoms and keep one orientation. These
are geometrically measurable paths, **not** a classification of rotatable bonds.
Ring and conjugation constraints require additional chemical information.

In [2]:
paths = msm.topology.get_covalent_paths(molsys, path=['all'] * 4)
quartets = np.asarray([
    path for path in paths if len(set(path)) == 4 and path[0] < path[-1]], dtype=int)
angles = msm.structure.get_dihedral_angles(molsys, dihedral_quartets=quartets, pbc=False)
print('Declared-connectivity quartets:', quartets.shape[0])
print('Torsions (degrees):', puw.get_value(angles, to_unit='degrees'))

Declared-connectivity quartets: 47
Torsions (degrees): [[ 6.01247232e-02 -1.22107140e-02 -4.03678286e+01  1.39536268e+02
   4.59313785e-02  1.79978645e+02 -3.06496743e-02  1.79986258e+02
  -1.79990228e+02 -6.40408127e-02 -1.79945157e+02  1.39551244e+02
  -4.05446592e+01 -1.25489078e+02  1.19260641e+02 -1.38969796e+00
  -6.14586818e+01  1.79841789e+02  1.75898135e+02 -8.95132579e+01
   9.80414784e+01 -1.42891250e+02  1.02532830e+02 -1.79275325e+02
  -1.08955746e+02 -1.49509658e+02  1.39630593e+02 -4.09756164e+01
  -6.89051040e+01  2.72056647e+01 -2.72168821e-01  1.81192285e+01
  -1.62486981e+02 -1.70765918e+01 -2.74173376e+01 -1.79658498e+02
   7.52437912e+00 -1.79985316e+02  1.79884754e+02 -1.49295494e+02
   3.08971515e+01  3.90723691e-02  7.73967339e-02 -1.78254847e-01
  -1.40452058e-01  1.20007317e-01  8.24505561e-02]]


(course-antiviral-conformational-engineering-challenge)=
## 🏆 Challenge: Preparing an Edit Safely

Check whether a proposed central bond is acyclic, genuinely rotatable and
supported by complete chemical connectivity. Determine its moving atoms before
using {func}`molsysmt.structure.shift_dihedral_angles`. This notebook deliberately
performs **measurements only**: missing connectivity in a PDB ligand must not be
mistaken for permission to rotate a disconnected fragment.

:::{key-takeaway}
:class: dropdown
Explicit quartets identify the measured torsion; explicit moving blocks identify
what an edit moves. A verified angle is a geometric result, not physical validation.
:::

(course-antiviral-conformational-engineering-see-also)=
:::{seealso}
:class: dropdown
{func}`molsysmt.topology.get_covalent_paths`,
{func}`molsysmt.structure.get_dihedral_angles`,
and [Module 16: Structural Operations](../Common_Core/16_Structural_Operations.ipynb).
:::